In [1]:
import os
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score
import numpy as np 
import joblib

In [2]:
left_features_df = pd.read_csv("dataset/processed/left_hand_features.csv")
right_features_df = pd.read_csv("dataset/processed/right_hand_features.csv")

In [3]:
def evaluate_svm(df, hand_label):
    X = df.drop(columns=['target', 'patient_id', 'hand'])
    y = df['target']

    feature_cols = X.columns

    # Stratified k-fold cross-validation
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    svm_pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('svm', SVC(
            kernel='rbf',
            C=0.5,
            gamma='scale',
            class_weight='balanced',
            probability=True,
            random_state=42
        ))
    ])

    # Cross-validated metrics
    scores = cross_validate(
        svm_pipeline, X, y, cv=skf,
        scoring={
            'accuracy': 'accuracy',
            'precision': 'precision',
            'recall': 'recall',
            'f1': 'f1',
            'roc_auc': 'roc_auc'
        }
    )

    print(f"\n===== {hand_label} Hand SVM Evaluation =====")
    print(f"Accuracy : {np.mean(scores['test_accuracy']):.2f}")
    print(f"Precision: {np.mean(scores['test_precision']):.2f}")
    print(f"Recall   : {np.mean(scores['test_recall']):.2f}")
    print(f"F1 Score : {np.mean(scores['test_f1']):.2f}")
    print(f"ROC-AUC  : {np.mean(scores['test_roc_auc']):.2f}")

    # Predictions for confusion matrix & report
    y_pred = cross_val_predict(svm_pipeline, X, y, cv=skf)
    y_prob = cross_val_predict(svm_pipeline, X, y, cv=skf, method='predict_proba')[:, 1]

    print("\nConfusion Matrix:")
    print(confusion_matrix(y, y_pred))

    print("\nClassification Report:")
    print(classification_report(y, y_pred))

    print(f"ROC-AUC (pooled): {roc_auc_score(y, y_prob):.2f}")

    # Safety uncertainty band
    def classify_with_uncertainty(p):
        if p < 0.40:
            return 0
        elif p > 0.50:
            return 1
        else:
            return -1   # unsure

    safe_preds = np.array([classify_with_uncertainty(p) for p in y_prob])
    unsure_rate = np.mean(safe_preds == -1)
    print(f"\nUncertainty Rate: {unsure_rate:.2f}")

    # ==============================
    # SAVE MODEL
    # ==============================
    model_path = os.path.join(f"models/{hand_label.lower()}_svm.pkl")

    # Fit the full pipeline on all data before saving
    svm_pipeline.fit(X, y)

    joblib.dump(svm_pipeline, model_path)

    print("\nSaved:")
    print(model_path)

# Evaluate both hands
evaluate_svm(left_features_df, "Left")
evaluate_svm(right_features_df, "Right")


===== Left Hand SVM Evaluation =====
Accuracy : 0.99
Precision: 0.96
Recall   : 1.00
F1 Score : 0.98
ROC-AUC  : 1.00

Confusion Matrix:
[[47  1]
 [ 0 18]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.98      0.99        48
           1       0.95      1.00      0.97        18

    accuracy                           0.98        66
   macro avg       0.97      0.99      0.98        66
weighted avg       0.99      0.98      0.98        66

ROC-AUC (pooled): 1.00

Uncertainty Rate: 0.02

Saved:
models/left_svm.pkl

===== Right Hand SVM Evaluation =====
Accuracy : 0.97
Precision: 0.90
Recall   : 1.00
F1 Score : 0.94
ROC-AUC  : 1.00

Confusion Matrix:
[[50  2]
 [ 0 14]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.96      0.98        52
           1       0.88      1.00      0.93        14

    accuracy                           0.97        66
   macro avg       